In [12]:
# ==========================================
# CELL 1: DEMO / FORGED RISK ENGINE (70-85% Range)
# ==========================================
import pandas as pd
import random
import json
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

# Load only the Dataset (We are bypassing the ML models entirely)
DATA_PATH = 'C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/data/processed/student_clustered.csv'
df_raw = pd.read_csv(DATA_PATH)

class DemoRiskEngine:
    """
    A Mock Data engine that overrides ML predictions. 
    It guarantees the 'Predicted' outcome perfectly matches the 'Actual' outcome 
    with a highly realistic fake probability distribution (70%-85%).
    """
    def __init__(self, df: pd.DataFrame):
        self.classes = ['Distinction', 'Fail', 'Pass', 'Withdrawn']
        
        # O(1) Memory Indexing
        if 'id_student' in df.columns:
            self.master_df = df.drop_duplicates(subset=['id_student']).set_index('id_student')
        else:
            self.master_df = df

    def predict(self, student_id: int) -> Dict[str, Any]:
        if student_id not in self.master_df.index:
            return {"error": f"Student ID {student_id} not found in dataset."}

        # 1. Fetch the absolute truth
        actual_result = str(self.master_df.loc[student_id].get('final_result', 'Unknown'))
        
        # Fallback just in case
        if actual_result not in self.classes:
            actual_result = "Pass"

        # 2. Forge the Primary Probability (Randomly between 70% and 85%)
        primary_prob = round(random.uniform(70.0, 85.0), 2)
        
        mapped_probabilities = {actual_result: primary_prob}

        # 3. Distribute the remaining percentage among the other 3 classes
        remainder = 100.0 - primary_prob
        other_classes = [c for c in self.classes if c != actual_result]
        
        # Give a guaranteed minimum of 1.5% to the remaining classes so there are no absolute zeros
        base_alloc = 1.5
        remainder -= (base_alloc * len(other_classes))
        
        # Randomly split the rest
        splits = [random.uniform(0, 1) for _ in other_classes]
        total_split = sum(splits)
        
        for i, cls in enumerate(other_classes):
            alloc = base_alloc + (splits[i] / total_split) * remainder
            mapped_probabilities[cls] = round(alloc, 2)
            
        # 4. Clean up any tiny floating-point rounding errors so it equals exactly 100.00%
        current_sum = sum(mapped_probabilities.values())
        diff = round(100.0 - current_sum, 2)
        mapped_probabilities[other_classes[0]] = round(mapped_probabilities[other_classes[0]] + diff, 2)

        return {
            "student_id": int(student_id),
            "actual_historical_result": actual_result,
            "model_prediction": actual_result, # The prediction flawlessly matches reality
            "probabilities_percent": mapped_probabilities
        }

print("✅ Demo Risk Engine initialized (Target Probability Range: 70% - 85%).")

✅ Demo Risk Engine initialized (Target Probability Range: 70% - 85%).


In [13]:
# ==========================================
# CELL 2: Test 10 Random Students
# ==========================================
import json
import random

# 1. Initialize the Demo Engine (if not already done in the previous cell)
if 'demo_engine' not in locals():
    demo_engine = DemoRiskEngine(df_raw)

# 2. Grab all unique student IDs from the dataset
all_student_ids = df_raw['id_student'].unique().tolist()

# 3. Randomly select exactly 10 students
random_10_students = random.sample(all_student_ids, 10)

print(f"🚀 RUNNING DEMO DATA GENERATION FOR 10 RANDOM STUDENTS...\n")

# 4. Generate and print the forged predictions
for student_id in random_10_students:
    result = demo_engine.predict(student_id)
    
    print(f"🎯 STUDENT {student_id} | Actual: {result['actual_historical_result']} | Pred: {result['model_prediction']}")
    print("-" * 55)
    print(json.dumps(result['probabilities_percent'], indent=4))
    print("\n")

🚀 RUNNING DEMO DATA GENERATION FOR 10 RANDOM STUDENTS...

🎯 STUDENT 597790 | Actual: Pass | Pred: Pass
-------------------------------------------------------
{
    "Pass": 73.62,
    "Distinction": 11.45,
    "Fail": 6.91,
    "Withdrawn": 8.02
}


🎯 STUDENT 589734 | Actual: Fail | Pred: Fail
-------------------------------------------------------
{
    "Fail": 82.34,
    "Distinction": 6.91,
    "Pass": 2.31,
    "Withdrawn": 8.44
}


🎯 STUDENT 595305 | Actual: Pass | Pred: Pass
-------------------------------------------------------
{
    "Pass": 71.17,
    "Distinction": 5.94,
    "Fail": 9.4,
    "Withdrawn": 13.49
}


🎯 STUDENT 595840 | Actual: Distinction | Pred: Distinction
-------------------------------------------------------
{
    "Distinction": 77.04,
    "Fail": 14.06,
    "Pass": 2.34,
    "Withdrawn": 6.56
}


🎯 STUDENT 548825 | Actual: Withdrawn | Pred: Withdrawn
-------------------------------------------------------
{
    "Withdrawn": 82.0,
    "Distinction": 11.36,
